# Subtask 2: Taiji MBHB Parameter Estimation

This notebook implements UCAS 2026 Task 5, subtask 2. It is structured around `TriangleDataCenter/Triangle-BBH` Example 4: `4_TDC_Verification_MBHB_Search_and_Estimation(CPU).ipynb`.

Goals:

1. Reproduce Triangle-BBH Example 4 as the baseline.
2. Record and understand the TDI data, Fourier-domain response, MBHB waveform, and Bayesian inference flow.
3. Modify the parameter-estimation data window to 5 days: 4 days before `coalescence_time` and 1 day after it.
4. Re-run search and inference, then compare the 5-day posterior against the baseline posterior.
5. Save figures, chains, and summary tables under `figures/task5_subtask2/` and `results/task5_subtask2/`.

By default this notebook only runs lightweight checks. Expensive search and sampler blocks are controlled by `RUN_SEARCH`, `RUN_SAMPLER`, and `RUN_FIVE_DAY`.


## 0. Execution Checklist

This notebook is organized as a step-by-step execution checklist for subtask 2:

- [ ] Record environment, third-party repository commits, and waveform backend.
- [ ] Configure TDC data paths, parameter paths, and Triangle orbit path.
- [ ] Import dependencies matching Triangle-BBH Example 4 and verify the CPU/WF4Py route.
- [ ] Load TDC II MBHB TDI data and injected parameters.
- [ ] Reproduce Example 4 baseline data slicing, FFT, PSD, model setup, search, and sampler flow.
- [ ] Save baseline diagnostic figures, corner plot, and parameter table.
- [ ] Modify the data window to `tc - 4 days` through `tc + 1 day`.
- [ ] Rebuild time-domain data, frequency-domain data, PSD, frequency grid, and likelihood for the new window.
- [ ] Run a 5-day smoke test before the full search and sampler.
- [ ] Compare baseline and 5-day posterior medians, 90% credible intervals, and figures.
- [ ] Add final subtask 2 results and links to the README.


## 1. Environment and Reproducibility Record

Run this section before any analysis. If the official Linux or WSL environment is used later, rerun this cell and keep the output in the notebook.


In [1]:
from __future__ import annotations

import importlib
import os
import platform
import subprocess
import sys
from pathlib import Path

# Keep numerical libraries from oversubscribing CPU threads during likelihood calls.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

TRIANGLE_BBH_DIR = REPO_ROOT / "external" / "Triangle-BBH"
TRIANGLE_SIM_DIR = REPO_ROOT / "external" / "Triangle-Simulator"
FIGURE_DIR = REPO_ROOT / "figures" / "task5_subtask2"
RESULT_DIR = REPO_ROOT / "results" / "task5_subtask2"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Repo root:", REPO_ROOT)
print("Triangle-BBH exists:", TRIANGLE_BBH_DIR.exists())
print("Triangle-Simulator exists:", TRIANGLE_SIM_DIR.exists())

def git_commit(path: Path) -> str:
    if not path.exists():
        return "missing"
    try:
        return subprocess.check_output(
            ["git", "-C", str(path), "rev-parse", "--short", "HEAD"],
            text=True,
            stderr=subprocess.STDOUT,
        ).strip()
    except Exception as exc:
        return f"unavailable: {exc}"

print("Triangle-BBH commit:", git_commit(TRIANGLE_BBH_DIR))
print("Triangle-Simulator commit:", git_commit(TRIANGLE_SIM_DIR))


Python: 3.12.13 (main, Mar  3 2026, 15:01:35) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
Repo root: C:\Users\雷畅\Documents\Codex\2026-05-19\files-mentioned-by-the-user-2026\task5-lisa-taiji
Triangle-BBH exists: True
Triangle-Simulator exists: True
Triangle-BBH commit: unavailable: Command '['git', '-C', 'C:\\Users\\雷畅\\Documents\\Codex\\2026-05-19\\files-mentioned-by-the-user-2026\\task5-lisa-taiji\\external\\Triangle-BBH', 'rev-parse', '--short', 'HEAD']' returned non-zero exit status 128.
Triangle-Simulator commit: unavailable: Command '['git', '-C', 'C:\\Users\\雷畅\\Documents\\Codex\\2026-05-19\\files-mentioned-by-the-user-2026\\task5-lisa-taiji\\external\\Triangle-Simulator', 'rev-parse', '--short', 'HEAD']' returned non-zero exit status 128.


## 2. Runtime Switches and Paths

Set local TDC data paths here. Large HDF5 files must not be committed to GitHub.

The official Example 4 commonly uses these files:

- `0_2_MBHB_TDIXYZ.h5`
- `0_2_MBHB_parameters.h5`

If your TDC II files are stored elsewhere, only change the paths below.


In [2]:
# Heavy execution switches. Keep these False while checking paths and shapes.
RUN_BASELINE = False
RUN_SEARCH = False
RUN_SAMPLER = False
RUN_FIVE_DAY = False

DAY = 24 * 3600
YEAR = 365 * DAY

# TODO: Change these paths to your local TDC II files before running the full workflow.
TDC_ROOT_DIR = Path(r"E:\BaiduNetdiskDownload\TDCData")
DATA_DIR = TDC_ROOT_DIR / "0_2_MBHB_TDIXYZ.h5"
PARAM_DIR = TDC_ROOT_DIR / "0_2_MBHB_parameters.h5"
ORBIT_DIR = TRIANGLE_SIM_DIR / "orbit" / "orbit_file"

print("DATA_DIR:", DATA_DIR, DATA_DIR.exists())
print("PARAM_DIR:", PARAM_DIR, PARAM_DIR.exists())
print("ORBIT_DIR:", ORBIT_DIR, ORBIT_DIR.exists())


DATA_DIR: E:\BaiduNetdiskDownload\TDCData\0_2_MBHB_TDIXYZ.h5 False
PARAM_DIR: E:\BaiduNetdiskDownload\TDCData\0_2_MBHB_parameters.h5 False
ORBIT_DIR: C:\Users\雷畅\Documents\Codex\2026-05-19\files-mentioned-by-the-user-2026\task5-lisa-taiji\external\Triangle-Simulator\orbit\orbit_file False


## 3. Imports Matching Triangle-BBH Example 4

Keep these imports close to the official notebook. On the current Windows CPU setup, `cupy` and `BBHx` are not required; the intended route is WF4Py CPU.


In [3]:
import json
import math
import multiprocessing
import pickle
import warnings

import bilby
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.optimize import differential_evolution
from tqdm import tqdm

from Triangle.Constants import *
from Triangle.Orbit import *
from Triangle.Noise import *
from Triangle.FFTTools import *
from Triangle.TDI import *
from Triangle.Data import *
from Triangle_BBH.Waveform import *
from Triangle_BBH.Response import *
from Triangle_BBH.Utils import *
from Triangle_BBH.Fisher import *

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

print("Imports OK")
print("bilby:", getattr(bilby, "__version__", "unknown"))


no cupy 
no cupy
no BBHx waveform
Imports OK
bilby: 1.0.0: release


C:\Users\雷畅\AppData\Local\Temp\ipykernel_37508\2421820933.py:16: RuntimeWarning: Skipping Triangle.GW: missing optional dependency lal
  from Triangle.Constants import *
C:\Users\雷畅\AppData\Local\Temp\ipykernel_37508\2421820933.py:16: RuntimeWarning: Skipping Triangle.Glitch: missing optional dependency lal
  from Triangle.Constants import *
C:\Users\雷畅\AppData\Local\Temp\ipykernel_37508\2421820933.py:16: RuntimeWarning: Skipping Triangle.Interferometer: missing optional dependency lal
  from Triangle.Constants import *


## 4. Utility Functions

These helpers keep baseline and 5-day runs comparable. They do not change the official algorithm; they only organize path checks, figure saving, data slicing, and summary tables.


In [4]:
def save_current_figure(filename: str) -> Path:
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved: {path.relative_to(REPO_ROOT)}")
    return path


def require_file(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


def summarize_array(name: str, arr: np.ndarray) -> dict:
    arr = np.asarray(arr)
    summary = {
        "name": name,
        "shape": arr.shape,
        "dtype": str(arr.dtype),
        "finite_fraction": float(np.isfinite(arr).mean()) if arr.size else np.nan,
        "min": float(np.nanmin(arr)) if arr.size else np.nan,
        "max": float(np.nanmax(arr)) if arr.size else np.nan,
    }
    return summary


def print_h5_tree(path: Path, max_items: int = 80) -> None:
    require_file(path, "HDF5 file")
    count = 0
    with h5py.File(path, "r") as h5:
        def visitor(name, obj):
            nonlocal count
            if count >= max_items:
                return
            if isinstance(obj, h5py.Dataset):
                print(f"DATASET /{name}: shape={obj.shape}, dtype={obj.dtype}")
            else:
                print(f"GROUP   /{name}")
            count += 1
        h5.visititems(visitor)
    if count >= max_items:
        print(f"... stopped after {max_items} items")


def posterior_summary(samples: pd.DataFrame, parameters: list[str]) -> pd.DataFrame:
    rows = []
    for p in parameters:
        if p not in samples:
            continue
        q05, q50, q95 = np.percentile(samples[p], [5, 50, 95])
        rows.append({
            "parameter": p,
            "median": q50,
            "ci90_low": q05,
            "ci90_high": q95,
            "ci90_width": q95 - q05,
        })
    return pd.DataFrame(rows)


## 5. Inspect TDC Data and Parameters

Run this after the TDC II files are downloaded. The first goal is only to confirm the HDF5 structure, not to run inference.


In [5]:
if DATA_DIR.exists():
    print("Data HDF5 structure:")
    print_h5_tree(DATA_DIR)
else:
    print("DATA_DIR does not exist yet. Update the path in Section 2 before running data cells.")

if PARAM_DIR.exists():
    print("\nParameter HDF5 structure:")
    print_h5_tree(PARAM_DIR)
else:
    print("PARAM_DIR does not exist yet. Update the path in Section 2 before running parameter cells.")


DATA_DIR does not exist yet. Update the path in Section 2 before running data cells.
PARAM_DIR does not exist yet. Update the path in Section 2 before running parameter cells.


## 6. Load Data Following Example 4

This mirrors the official Example 4: read the TDI XYZ file, read injected parameters, convert XYZ to A/E/T, and keep A/E for the parameter-estimation pipeline.


In [6]:
read_dict = None
injected_parameters = None

def load_tdc_inputs(data_path: Path, param_path: Path):
    require_file(data_path, "TDC data file")
    require_file(param_path, "TDC parameter file")

    with h5py.File(data_path, "r") as h5file:
        data = read_dict_from_h5(h5file["/"])

    with h5py.File(param_path, "r") as h5file:
        params = read_dict_from_h5(h5file["/"])

    return data, params

if DATA_DIR.exists() and PARAM_DIR.exists():
    read_dict, injected_parameters = load_tdc_inputs(DATA_DIR, PARAM_DIR)
    print("Loaded data keys:", read_dict.keys())
    print("Loaded parameter keys:", injected_parameters.keys())
else:
    print("Skipping load because TDC files are not configured yet.")


Skipping load because TDC files are not configured yet.


## 7. Baseline Data Preparation

This section should reproduce the official Example 4 baseline before any 5-day modification. The important quantities to record are:

- original time span and sampling cadence;
- channel convention and sign convention;
- baseline slice start/end used by Example 4;
- frequency range and frequency resolution;
- PSD estimation interval;
- dimensions of `data_frequency`, `data_channels_fd`, and `InvCovMat`.


In [7]:
channel_names = ["A2", "E2"]

def build_aet_channels(read_dict: dict):
    data_time = np.asarray(read_dict["time"])
    A2_td, E2_td, _ = AETfromXYZ(
        read_dict["XYZ"]["X2"],
        read_dict["XYZ"]["Y2"],
        read_dict["XYZ"]["Z2"],
    )
    # Official Example 4 uses a minus sign for the A/E data channels.
    data_channels_td = -np.array([A2_td, E2_td])
    dt = float(np.median(np.diff(data_time)))
    return data_time, data_channels_td, dt

if read_dict is not None:
    data_time, data_channels_td, dt = build_aet_channels(read_dict)
    print("time:", summarize_array("time", data_time))
    print("channels:", summarize_array("A/E time domain", data_channels_td))
    print("dt:", dt, "seconds")
else:
    data_time = data_channels_td = dt = None
    print("Run Section 6 after configuring TDC files.")


Run Section 6 after configuring TDC files.


### 7.1 Baseline Quick-Look Plot

This plot verifies that the A/E streams are loaded and that the time axis uses days. Save it as a baseline diagnostic figure.


In [8]:
if data_time is not None:
    plt.figure(figsize=(10, 4))
    for i, name in enumerate(channel_names):
        plt.plot(data_time / DAY, data_channels_td[i], lw=0.8, label=name)
    plt.xlabel("Time (day)")
    plt.ylabel("TDI")
    plt.legend()
    save_current_figure("01_baseline_aet_timeseries.png")
else:
    print("No data loaded yet.")


No data loaded yet.


## 8. Shared Window Builder: Baseline and 5-Day Runs

All downstream arrays must be rebuilt when the observation window changes. This is the key rigor point for subtask 2: do not only change `Tobs`; update the time-domain slice, FFT, PSD, frequency grid, waveform response, and likelihood together.


In [9]:
def slice_time_domain_window(data_time, data_channels_td, t_start, t_end):
    mask = (data_time >= t_start) & (data_time <= t_end)
    if mask.sum() < 8:
        raise ValueError("Selected window contains too few samples.")
    return data_time[mask], data_channels_td[:, mask]


def fft_channels(data_channels_window, dt, window_type="tukey"):
    freqs = None
    channels_fd = []
    for i in range(len(data_channels_window)):
        ff, xf = FFT_window(
            data_array=data_channels_window[i],
            fsample=1.0 / dt,
            window_type=window_type,
        )
        freqs = ff
        channels_fd.append(xf)
    return np.asarray(freqs), np.asarray(channels_fd)


def estimate_psd_from_sidebands(A2_td, E2_td, data_time, signal_t_start, signal_t_end, dt):
    silent_mask = (data_time < signal_t_start) | (data_time > signal_t_end)
    if silent_mask.sum() < 16:
        raise ValueError("Not enough silent data to estimate PSD.")

    ff_a, A2_PSD = PSD_window(
        data_array=A2_td[silent_mask],
        fsample=1.0 / dt,
        window_type="hann",
        nbin=20,
    )
    ff_e, E2_PSD = PSD_window(
        data_array=E2_td[silent_mask],
        fsample=1.0 / dt,
        window_type="hann",
        nbin=20,
    )
    return ff_a, np.asarray([A2_PSD, E2_PSD])


def restrict_frequency_band(data_frequency, data_channels_fd, psd_channels, fmin=0.5e-4, fmax=1e-2):
    freq_idx = np.where((data_frequency >= fmin) & (data_frequency <= fmax))[0]
    if len(freq_idx) == 0:
        raise ValueError("Frequency band is empty. Check dt, window length, fmin, and fmax.")
    return data_frequency[freq_idx], data_channels_fd[:, freq_idx], psd_channels[:, freq_idx]


## 9. Reproduce Official Example 4 Baseline

The baseline should be run first with settings as close as possible to Example 4. The exact official cells are intentionally not pasted wholesale here; instead, each cell below maps to a block in Example 4 while keeping outputs organized in this repository.

Fill `BASELINE_T_START` and `BASELINE_T_END` from the official Example 4 after checking the loaded data and coalescence time.


In [10]:
BASELINE_T_START = None  # TODO: set from official Example 4 / data inspection.
BASELINE_T_END = None    # TODO: set from official Example 4 / data inspection.
FMIN = 0.5e-4
FMAX = 1e-2

baseline = {}

if RUN_BASELINE:
    if data_time is None or injected_parameters is None:
        raise RuntimeError("Load TDC data and parameters before running baseline.")
    if BASELINE_T_START is None or BASELINE_T_END is None:
        raise RuntimeError("Set BASELINE_T_START and BASELINE_T_END before running baseline.")

    base_time, base_td = slice_time_domain_window(data_time, data_channels_td, BASELINE_T_START, BASELINE_T_END)
    base_freq, base_fd = fft_channels(base_td, dt)

    psd_freq, psd_channels_raw = estimate_psd_from_sidebands(
        data_channels_td[0], data_channels_td[1], data_time, BASELINE_T_START, BASELINE_T_END, dt
    )

    psd_channels = np.vstack([
        np.interp(base_freq, psd_freq, psd_channels_raw[0]),
        np.interp(base_freq, psd_freq, psd_channels_raw[1]),
    ])
    base_freq, base_fd, psd_channels = restrict_frequency_band(base_freq, base_fd, psd_channels, FMIN, FMAX)
    InvCovMat = np.array([np.diag(1.0 / psd_channels[:, i]) for i in range(len(base_freq))])

    baseline.update(
        time=base_time,
        data_td=base_td,
        frequency=base_freq,
        data_fd=base_fd,
        psd=psd_channels,
        inverse_covariance=InvCovMat,
    )
    print("Baseline frequency shape:", base_freq.shape)
    print("Baseline data_fd shape:", base_fd.shape)
    print("Baseline InvCovMat shape:", InvCovMat.shape)
else:
    print("RUN_BASELINE=False. Baseline arrays will not be built in this lightweight pass.")


RUN_BASELINE=False. Baseline arrays will not be built in this lightweight pass.


### 9.1 Baseline Frequency-Domain Diagnostic


In [11]:
if baseline:
    plt.figure(figsize=(10, 4))
    for i, name in enumerate(channel_names):
        plt.loglog(baseline["frequency"], np.abs(baseline["data_fd"][i]), label=f"{name} data")
        plt.loglog(baseline["frequency"], np.sqrt(baseline["psd"][i]), ls="--", label=f"{name} sqrt(PSD)")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Amplitude")
    plt.legend(ncol=2)
    save_current_figure("02_baseline_frequency_psd.png")
else:
    print("Build baseline arrays first.")


Build baseline arrays first.


## 10. Model Setup: Orbit, Waveform, Response, Noise

This follows Example 4: initialize the Taiji orbit, WF4Py frequency-domain waveform generator, TDI response generator, and response keyword dictionaries.


In [12]:
orbit = None
WFG = None
FDTDI = None
response_kwargs_direct = None
response_kwargs_intrinsic = None

if RUN_BASELINE or RUN_SEARCH or RUN_SAMPLER or RUN_FIVE_DAY:
    require_file(Path(ORBIT_DIR), "Orbit directory or file")
    orbit = Orbit(OrbitDir=str(ORBIT_DIR))
    WFG = WaveformGeneratorFRef(mode="primary")
    FDTDI = FDTDIResponseGeneratorFRef(
        det="Taiji",
        orbit=orbit,
        waveform_generator=WFG,
        use_gpu=False,
    )
    print("Model generators initialized.")
else:
    print("Model setup skipped because heavy switches are False.")


Model setup skipped because heavy switches are False.


## 11. Search High-Likelihood Parameters with F-statistics

This block corresponds to the Example 4 differential-evolution search. It should be run after the baseline data arrays and model generators are ready.


In [13]:
searched_parameters = None
searched_wf = None
Fstat = None
DE_result = None

if RUN_SEARCH:
    if not baseline or FDTDI is None:
        raise RuntimeError("Build baseline data and initialize FDTDI before search.")

    data_frequency = baseline["frequency"]
    data_channels_fd = baseline["data_fd"]
    InvCovMat = baseline["inverse_covariance"]

    # TODO: copy exact priors from official Example 4 after confirming parameter names and units.
    lgMc_prior = [5.0, 7.0]
    q_prior = [0.01, 0.99]
    s1_prior = [-0.9, 0.9]
    s2_prior = [-0.9, 0.9]

    # TODO: define response_kwargs_direct / response_kwargs_intrinsic exactly as in Example 4.
    # Fstat = FstatisticsFref(...)
    # DE_result = differential_evolution(...)
    # searched_parameters = ...
    raise NotImplementedError("Fill this block from Example 4 after baseline arrays are verified.")
else:
    print("RUN_SEARCH=False. Search block is scaffolded but not executed.")


RUN_SEARCH=False. Search block is scaffolded but not executed.


## 12. Fisher Analysis Around Searched Parameters

Use this section to reproduce Example 4 Fisher diagnostics and to set narrow priors for the final sampler. Save the Fisher parameter-error table for comparison with posterior widths.


In [14]:
FIM = None

if RUN_SEARCH and searched_parameters is not None:
    # TODO: define fisher_waveform_wrapper and run FisherMatrix exactly as in Example 4.
    # FIM = FisherMatrix(...)
    # fisher_table = pd.DataFrame(...)
    # fisher_table.to_csv(RESULT_DIR / "baseline_fisher_errors.csv", index=False)
    raise NotImplementedError("Add Fisher analysis after search is working.")
else:
    print("Fisher block waiting for searched_parameters.")


Fisher block waiting for searched_parameters.


## 13. Likelihood and Bilby Wrapper

This is adapted from Example 4. The same wrapper should be reused for baseline and 5-day runs after rebuilding the `Likelihood` object with the corresponding data arrays.


In [15]:
class BilbyLikelihoodWrapper(bilby.Likelihood):
    def __init__(self, like_object, like_type="heterodyned"):
        super().__init__(
            parameters={
                "chirp_mass": None,
                "mass_ratio": None,
                "spin_1z": None,
                "spin_2z": None,
                "reference_time": None,
                "reference_phase": None,
                "luminosity_distance": None,
                "inclination": None,
                "longitude": None,
                "latitude": None,
                "psi": None,
            }
        )
        self.like_object = like_object
        self.like_type = like_type

    def log_likelihood(self):
        parameter_array = ParamDict2ParamArrFref(self.parameters)
        if self.like_type == "heterodyned":
            return self.like_object.het_log_like(parameter_array=parameter_array)
        return self.like_object.full_log_like(parameter_array=parameter_array)


## 14. Baseline Sampler Run

Run a smoke test first, then increase sampler settings. Keep the full run output under `results/task5_subtask2/baseline_samples/` and figures under `figures/task5_subtask2/`.


In [16]:
baseline_result = None
baseline_summary = None

BASELINE_SAMPLER_CONFIG = {
    "sampler": "nessai",
    "nlive": 1200,
    "stopping": 0.1,
    "label": "baseline_example4",
    "outdir": str(RESULT_DIR / "baseline_samples"),
}

if RUN_SAMPLER:
    # TODO: construct priors from searched_parameters and FIM as in Example 4.
    # BLike = BilbyLikelihoodWrapper(Like, like_type="heterodyned")
    # baseline_result = bilby.run_sampler(...)
    # baseline_result.plot_corner(save=True)
    raise NotImplementedError("Sampler run will be enabled after priors and likelihood are verified.")
else:
    print("RUN_SAMPLER=False. Baseline sampler block is scaffolded but not executed.")


RUN_SAMPLER=False. Baseline sampler block is scaffolded but not executed.


## 15. Define the Required 5-Day Window

The task requirement is:

$$
t_{start} = t_c - 4\,\mathrm{days}, \qquad
t_{end} = t_c + 1\,\mathrm{day}, \qquad
T_{obs} = 5\,\mathrm{days}.
$$

Critical checks:

- `coalescence_time` unit must be confirmed from the parameter file and Example 4.
- The new time-domain slice must contain both A and E channels.
- FFT, PSD interpolation, frequency band restriction, response generator, and likelihood must all use the new frequency grid.


In [17]:
def get_coalescence_time_seconds(injected_parameters: dict) -> float:
    tc = float(injected_parameters["coalescence_time"])
    # Example 4 often stores coalescence_time in days and multiplies by DAY for orbit calls.
    # If tc is already larger than one year in seconds, treat it as seconds; otherwise treat it as days.
    if tc > 10 * YEAR:
        return tc
    return tc * DAY

five_day = {}

if injected_parameters is not None:
    tc_seconds = get_coalescence_time_seconds(injected_parameters)
    FIVE_DAY_T_START = tc_seconds - 4 * DAY
    FIVE_DAY_T_END = tc_seconds + 1 * DAY
    FIVE_DAY_TOBS = FIVE_DAY_T_END - FIVE_DAY_T_START
    print("tc_seconds:", tc_seconds)
    print("five-day window start/end/duration:", FIVE_DAY_T_START, FIVE_DAY_T_END, FIVE_DAY_TOBS)
else:
    tc_seconds = FIVE_DAY_T_START = FIVE_DAY_T_END = FIVE_DAY_TOBS = None
    print("Load injected_parameters before defining the 5-day window.")


Load injected_parameters before defining the 5-day window.


## 16. Build 5-Day Data Arrays

This block is the core modification relative to Example 4. It should be kept separate from baseline so that both results can be compared and audited.


In [18]:
if RUN_FIVE_DAY:
    if data_time is None or FIVE_DAY_T_START is None:
        raise RuntimeError("Load data and injected parameters before building the 5-day window.")

    five_time, five_td = slice_time_domain_window(data_time, data_channels_td, FIVE_DAY_T_START, FIVE_DAY_T_END)
    five_freq, five_fd = fft_channels(five_td, dt)

    psd_freq, psd_channels_raw = estimate_psd_from_sidebands(
        data_channels_td[0], data_channels_td[1], data_time, FIVE_DAY_T_START, FIVE_DAY_T_END, dt
    )
    five_psd = np.vstack([
        np.interp(five_freq, psd_freq, psd_channels_raw[0]),
        np.interp(five_freq, psd_freq, psd_channels_raw[1]),
    ])
    five_freq, five_fd, five_psd = restrict_frequency_band(five_freq, five_fd, five_psd, FMIN, FMAX)
    five_inv_cov = np.array([np.diag(1.0 / five_psd[:, i]) for i in range(len(five_freq))])

    five_day.update(
        time=five_time,
        data_td=five_td,
        frequency=five_freq,
        data_fd=five_fd,
        psd=five_psd,
        inverse_covariance=five_inv_cov,
    )

    print("5-day time span (days):", (five_time[-1] - five_time[0]) / DAY)
    print("5-day df:", np.median(np.diff(five_freq)))
    print("Expected df=1/Tobs:", 1.0 / (five_time[-1] - five_time[0]))
    print("5-day data_fd shape:", five_fd.shape)
else:
    print("RUN_FIVE_DAY=False. 5-day arrays will not be built in this lightweight pass.")


RUN_FIVE_DAY=False. 5-day arrays will not be built in this lightweight pass.


### 16.1 5-Day Frequency-Domain Diagnostic


In [19]:
if five_day:
    plt.figure(figsize=(10, 4))
    for i, name in enumerate(channel_names):
        plt.loglog(five_day["frequency"], np.abs(five_day["data_fd"][i]), label=f"{name} data")
        plt.loglog(five_day["frequency"], np.sqrt(five_day["psd"][i]), ls="--", label=f"{name} sqrt(PSD)")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Amplitude")
    plt.legend(ncol=2)
    save_current_figure("03_five_day_frequency_psd.png")
else:
    print("Build five_day arrays first.")


Build five_day arrays first.


## 17. 5-Day Search, Fisher, Likelihood, and Sampler

Repeat the Example 4 search and sampler using the 5-day arrays. Do not reuse baseline likelihood objects after the window change.


In [20]:
five_day_result = None
five_day_summary = None

FIVE_DAY_SAMPLER_CONFIG = {
    "sampler": "nessai",
    "nlive": 1200,
    "stopping": 0.1,
    "label": "five_day_window",
    "outdir": str(RESULT_DIR / "five_day_samples"),
}

if RUN_FIVE_DAY and RUN_SEARCH:
    # TODO: rerun F-statistics search on five_day["frequency"], five_day["data_fd"], five_day["inverse_covariance"].
    # TODO: rebuild Fisher matrix around the five-day searched parameters.
    # TODO: rebuild Likelihood(...), prepare heterodyned log-likelihood, rerun Bilby sampler.
    raise NotImplementedError("Add 5-day search/sampler after baseline is verified.")
else:
    print("5-day inference block is scaffolded. Enable RUN_FIVE_DAY and RUN_SEARCH after baseline is reproducible.")


5-day inference block is scaffolded. Enable RUN_FIVE_DAY and RUN_SEARCH after baseline is reproducible.


## 18. Baseline vs 5-Day Comparison

The final report should compare medians and 90% credible intervals for key parameters. Width ratios below 1 mean the 5-day window gives tighter posterior constraints.


In [21]:
PARAMETERS_TO_COMPARE = [
    "chirp_mass",
    "mass_ratio",
    "spin_1z",
    "spin_2z",
    "reference_time",
    "reference_phase",
    "luminosity_distance",
    "inclination",
    "longitude",
    "latitude",
    "psi",
]

def compare_summaries(baseline_summary: pd.DataFrame, five_day_summary: pd.DataFrame) -> pd.DataFrame:
    base = baseline_summary.add_prefix("baseline_").rename(columns={"baseline_parameter": "parameter"})
    five = five_day_summary.add_prefix("five_day_").rename(columns={"five_day_parameter": "parameter"})
    merged = pd.merge(base, five, on="parameter", how="outer")
    merged["ci90_width_ratio_5day_over_baseline"] = (
        merged["five_day_ci90_width"] / merged["baseline_ci90_width"]
    )
    return merged

if baseline_summary is not None and five_day_summary is not None:
    comparison = compare_summaries(baseline_summary, five_day_summary)
    comparison.to_csv(RESULT_DIR / "baseline_vs_five_day_parameter_summary.csv", index=False)
    display(comparison)
else:
    comparison = pd.DataFrame(
        columns=[
            "parameter",
            "baseline_median",
            "baseline_ci90_low",
            "baseline_ci90_high",
            "five_day_median",
            "five_day_ci90_low",
            "five_day_ci90_high",
            "ci90_width_ratio_5day_over_baseline",
        ]
    )
    display(comparison)


,parameter,baseline_median,baseline_ci90_low,baseline_ci90_high,five_day_median,five_day_ci90_low,five_day_ci90_high,ci90_width_ratio_5day_over_baseline


## 19. Figures and Result Manifest

Keep a small manifest of generated outputs so README links can be added without hunting through folders.


In [22]:
manifest = {
    "baseline_timeseries": "figures/task5_subtask2/01_baseline_aet_timeseries.png",
    "baseline_frequency_psd": "figures/task5_subtask2/02_baseline_frequency_psd.png",
    "five_day_frequency_psd": "figures/task5_subtask2/03_five_day_frequency_psd.png",
    "baseline_corner": "figures/task5_subtask2/04_baseline_corner.png",
    "five_day_corner": "figures/task5_subtask2/05_five_day_corner.png",
    "comparison_table": "results/task5_subtask2/baseline_vs_five_day_parameter_summary.csv",
}
manifest_path = RESULT_DIR / "manifest.json"
with manifest_path.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(json.dumps(manifest, indent=2, ensure_ascii=False))


{
  "baseline_timeseries": "figures/task5_subtask2/01_baseline_aet_timeseries.png",
  "baseline_frequency_psd": "figures/task5_subtask2/02_baseline_frequency_psd.png",
  "five_day_frequency_psd": "figures/task5_subtask2/03_five_day_frequency_psd.png",
  "baseline_corner": "figures/task5_subtask2/04_baseline_corner.png",
  "five_day_corner": "figures/task5_subtask2/05_five_day_corner.png",
  "comparison_table": "results/task5_subtask2/baseline_vs_five_day_parameter_summary.csv"
}


## 20. Final Discussion Notes

Complete this section after the runs finish.

Questions to answer:

1. Did the baseline reproduce the main plots and parameter-estimation behavior of official Example 4?
2. Did the 5-day window change frequency resolution, PSD estimation, and likelihood stability?
3. Which parameters have clearly narrower 90% credible intervals? Which do not improve much?
4. If the result is unexpected, is the cause sampler convergence, simplified waveform modeling, noise estimation, signal overlap, or window choice?
5. Which figures and tables should be linked from the final README?

Draft conclusion:

- Baseline reproduction: TODO.
- Five-day modification: TODO.
- Posterior comparison: TODO.
- Limitations: TODO.
